In [1]:
import pandas as pd
import numpy as np
import os

In [6]:
df = pd.read_csv("../data/processed/cleaned_inventory.csv")

df.head()

,Date,Product_ID,Product_Name,Category,Brand,Current_Stock,Units_Sold,Unit_Price,Discount_Percent,Promotion,Holiday,Lead_Time_Days,Safety_Stock,Stockout
0,2022-02-17,P001,iPhone 13,Mobile,Apple,154.0,9,65000.0,0.0,0,0,7.0,15,0
1,2026-06-06,P008,Samsung 55-inch Smart TV,TV,Samsung,101.0,3,120000.0,0.0,1,0,14.0,5,0
2,2025-06-04,P006,HP Pavilion 15,Laptop,HP,101.0,5,95000.0,0.0,1,0,9.0,8,0
3,2022-11-14,P004,Redmi Note 13,Mobile,Xiaomi,91.0,10,32000.0,0.0,0,0,5.0,20,0
4,2025-08-08,P001,iPhone 13,Mobile,Apple,129.0,5,65000.0,0.0,0,0,7.0,15,0


In [7]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(["Product_ID", "Date"])

df.head()

,Date,Product_ID,Product_Name,Category,Brand,Current_Stock,Units_Sold,Unit_Price,Discount_Percent,Promotion,Holiday,Lead_Time_Days,Safety_Stock,Stockout
2678,2022-01-01,P001,iPhone 13,Mobile,Apple,211.0,7,65000.0,0.0,1,1,7.0,15,0
14707,2022-01-02,P001,iPhone 13,Mobile,Apple,125.0,6,65000.0,0.0,0,0,7.0,15,0
11497,2022-01-03,P001,iPhone 13,Mobile,Apple,147.0,7,65000.0,0.0,0,0,7.0,15,0
256,2022-01-04,P001,iPhone 13,Mobile,Apple,136.0,5,65000.0,0.0,0,0,7.0,15,0
9186,2022-01-05,P001,iPhone 13,Mobile,Apple,108.0,9,65000.0,0.0,0,0,7.0,15,0


In [8]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

df["Is_Weekend"] = (df["Day_of_Week"] >= 5).astype(int)

In [9]:
df["Lag_1_Day"] = (
    df.groupby("Product_ID")["Units_Sold"]
      .shift(1)
)

In [10]:
df["Lag_7_Days"] = (
    df.groupby("Product_ID")["Units_Sold"]
      .shift(7)
)

In [11]:
df["Lag_14_Days"] = (
    df.groupby("Product_ID")["Units_Sold"]
      .shift(14)
)

In [12]:
df["Rolling_7_Day"] = (
    df.groupby("Product_ID")["Units_Sold"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [13]:
df["Rolling_30_Day"] = (
    df.groupby("Product_ID")["Units_Sold"]
      .transform(lambda x: x.shift(1).rolling(30).mean())
)

In [14]:
df["Rolling_7_Day_Std"] = (
    df.groupby("Product_ID")["Units_Sold"]
      .transform(lambda x: x.shift(1).rolling(7).std())
)

In [15]:
df["Discount_Amount"] = (
    df["Unit_Price"] * df["Discount_Percent"] / 100
)

In [16]:
df["Final_Price"] = (
    df["Unit_Price"] - df["Discount_Amount"]
)

In [17]:
df.isnull().sum()

Date                   0
Product_ID             0
Product_Name           0
Category               0
Brand                  0
Current_Stock          0
Units_Sold             0
Unit_Price             0
Discount_Percent       0
Promotion              0
Holiday                0
Lead_Time_Days         0
Safety_Stock           0
Stockout               0
Year                   0
Month                  0
Day                    0
Day_of_Week            0
Week_of_Year           0
Is_Weekend             0
Lag_1_Day             10
Lag_7_Days            70
Lag_14_Days          140
Rolling_7_Day         70
Rolling_30_Day       300
Rolling_7_Day_Std     70
Discount_Amount        0
Final_Price            0
dtype: int64

In [18]:
feature_columns = [
    "Lag_1_Day",
    "Lag_7_Days",
    "Lag_14_Days",
    "Rolling_7_Day",
    "Rolling_30_Day",
    "Rolling_7_Day_Std"
]

df = df.dropna(subset=feature_columns).copy()

In [19]:
df.isnull().sum()

Date                 0
Product_ID           0
Product_Name         0
Category             0
Brand                0
Current_Stock        0
Units_Sold           0
Unit_Price           0
Discount_Percent     0
Promotion            0
Holiday              0
Lead_Time_Days       0
Safety_Stock         0
Stockout             0
Year                 0
Month                0
Day                  0
Day_of_Week          0
Week_of_Year         0
Is_Weekend           0
Lag_1_Day            0
Lag_7_Days           0
Lag_14_Days          0
Rolling_7_Day        0
Rolling_30_Day       0
Rolling_7_Day_Std    0
Discount_Amount      0
Final_Price          0
dtype: int64

In [20]:
print("Dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())

Dataset shape: (16440, 28)
Total missing values: 0


In [21]:
df["Rolling_30_Day"].fillna(df["Rolling_30_Day"].mean())

1176     7.000000
2534     6.866667
6565     6.933333
3862     6.966667
7366     7.000000
           ...   
15166    1.333333
9103     1.366667
3284     1.300000
5440     1.333333
7575     1.366667
Name: Rolling_30_Day, Length: 16440, dtype: float64

In [22]:
df.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.
